# Trusted Feature Engineering — Home Credit (AWS Glue)

Este notebook integra hallazgos de los EDAs:
- `EDA_train_payments.ipynb`
- `EDA-pos_cash-credit_card.ipynb`
- `EDA_bureau_bureau_balance.ipynb`

Objetivo: construir un pipeline de preparación de datos **cloud-ready** que lea datos desde Glue Catalog/S3, aplique limpieza y transformación, genere features de riesgo de crédito y guarde datasets listos para modelado SparkML en `refined`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType, NumericType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer

# Configuración base (editar según entorno)
BUCKET_NAME = 'hcdr'
RAW_PATH = 's3://{}/raw/'.format(BUCKET_NAME)
REFINED_PATH = 's3://{}/refined/'.format(BUCKET_NAME)
# Se mantiene por compatibilidad de nombre, pero no se usa para escritura real.
TRUSTED_PATH = REFINED_PATH
GLUE_RAW_DB = 'raw_db'
GLUE_TRUSTED_DB = 'trusted_db'

# Candidatos de base de datos en orden de prioridad
GLUE_TRUSTED_DB_CANDIDATES = [GLUE_TRUSTED_DB, 'hcdr_trusted']
GLUE_RAW_DB_CANDIDATES = [GLUE_RAW_DB]

# Si es False, se evita fallback a S3 para no disparar errores NoSuchBucket.
ENABLE_S3_FALLBACK = False

TABLES = [
    'application_train',
    'bureau',
    'bureau_balance',
    'previous_application',
    'pos_cash_balance',
    'credit_card_balance',
    'installments_payments'
]

DATASETS = {}
DATASET_PATHS = {}
WARNINGS = []

try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext
    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    print('GlueContext inicializado')
except Exception:
    spark = (
        SparkSession.builder
        .appName('Trusted_FeatureEngineering_HomeCredit_Glue')
        .config('spark.sql.adaptive.enabled', 'true')
        .config('spark.sql.shuffle.partitions', '240')
        .enableHiveSupport()
        .getOrCreate()
    )
    print('SparkSession fallback inicializada')

spark.sparkContext.setLogLevel('WARN')
spark.conf.set('spark.sql.shuffle.partitions', '240')


def sql(query, rows=20, truncate=False):
    df = spark.sql(query)
    df.show(rows, truncate=truncate)
    return df


def guardar_spark_en_trusted(df, nombre):
    # Redirección solicitada: no escribir en trusted, todo va a refined
    out = REFINED_PATH + nombre.strip('/') + '/'
    if '/trusted/' in out:
        raise ValueError('Bloqueado: salida a trusted detectada: {}'.format(out))
    df.write.mode('overwrite').parquet(out)
    DATASETS[nombre] = df
    DATASET_PATHS[nombre] = out
    print('Guardado refined (redirigido desde trusted) -> {}'.format(out))
    return df


def guardar_spark_en_refined(df, nombre):
    out = REFINED_PATH + nombre.strip('/') + '/'
    df.write.mode('overwrite').parquet(out)
    DATASETS[nombre] = df
    DATASET_PATHS[nombre] = out
    print('Guardado refined -> {}'.format(out))
    return df


def normalizar_columnas(df):
    cols = [c.lower().strip() for c in df.columns]
    return df.toDF(*cols)


def table_exists(database, table):
    try:
        return spark.catalog.tableExists('{}.{}'.format(database, table))
    except Exception:
        return False


def first_existing_table(table_name, db_candidates):
    for db in db_candidates:
        if table_exists(db, table_name):
            return db
    return None


def cargar_tabla_o_path(nombre_tabla, path_fallback):
    trusted_db = first_existing_table(nombre_tabla, GLUE_TRUSTED_DB_CANDIDATES)
    if trusted_db is not None:
        df = spark.table('{}.{}'.format(trusted_db, nombre_tabla))
        print('Tabla {} cargada desde glue_trusted ({})'.format(nombre_tabla, trusted_db))
        return normalizar_columnas(df)

    raw_db = first_existing_table(nombre_tabla, GLUE_RAW_DB_CANDIDATES)
    if raw_db is not None:
        df = spark.table('{}.{}'.format(raw_db, nombre_tabla))
        print('Tabla {} cargada desde glue_raw ({})'.format(nombre_tabla, raw_db))
        return normalizar_columnas(df)

    if not ENABLE_S3_FALLBACK:
        raise ValueError('Tabla {} no encontrada en Glue y ENABLE_S3_FALLBACK=False'.format(nombre_tabla))

    try:
        df = spark.read.parquet(path_fallback)
        origen = 's3_parquet'
    except Exception:
        df = (
            spark.read
            .option('header', 'true')
            .option('inferSchema', 'true')
            .csv(path_fallback)
        )
        origen = 's3_csv'

    df = normalizar_columnas(df)
    print('Tabla {} cargada desde {}'.format(nombre_tabla, origen))
    return df


def add_missing_columns(df, required_cols, default_value=None):
    out = df
    for c in required_cols:
        if c not in out.columns:
            out = out.withColumn(c, F.lit(default_value))
    return out


def warn_missing_columns(df, required_cols, table_name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msg = 'WARNING [{}] columnas faltantes: {}'.format(table_name, ', '.join(missing))
        WARNINGS.append(msg)
        print(msg)


def null_summary(df, top_n=80):
    exprs = []
    for c in df.columns:
        exprs.append(
            F.struct(
                F.lit(c).alias('columna'),
                F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).cast('double').alias('nulls'),
                F.count(F.lit(1)).cast('double').alias('total')
            ).alias(c)
        )
    stacked = df.select(*exprs)
    exploded = stacked.select(F.explode(F.array(*[F.col(c) for c in df.columns])).alias('x'))
    return (
        exploded
        .select(
            F.col('x.columna').alias('columna'),
            F.col('x.nulls').alias('nulls'),
            F.col('x.total').alias('total'),
            F.round(F.col('x.nulls') * 100.0 / F.col('x.total'), 4).alias('pct_nulls')
        )
        .orderBy(F.desc('pct_nulls'), F.desc('nulls'))
        .limit(top_n)
    )

print('Configuración y helpers listos')

In [ ]:
dfs = {}
for t in TABLES:
    fallback = RAW_PATH + t + '/'
    try:
        dft = cargar_tabla_o_path(t, fallback)
        dft = normalizar_columnas(dft)
        dft.createOrReplaceTempView(t)
        dfs[t] = dft
        print('{} -> rows: {:,} | cols: {}'.format(t, dft.count(), len(dft.columns)))
    except Exception as e:
        msg = 'WARNING: no se pudo cargar {}: {}'.format(t, str(e))
        WARNINGS.append(msg)
        print(msg)

application_train = dfs.get('application_train')
bureau = dfs.get('bureau')
bureau_balance = dfs.get('bureau_balance')
previous_application = dfs.get('previous_application')
pos_cash_balance = dfs.get('pos_cash_balance')
credit_card_balance = dfs.get('credit_card_balance')
installments_payments = dfs.get('installments_payments')

print('Tablas cargadas:', ', '.join(sorted(dfs.keys())))

In [ ]:
if application_train is None:
    raise ValueError('application_train es obligatoria para construir master_dataset')

app = normalizar_columnas(application_train)
required_app_cols = [
    'sk_id_curr', 'target', 'days_birth', 'days_employed', 'amt_credit',
    'amt_income_total', 'amt_annuity', 'amt_goods_price', 'cnt_fam_members', 'code_gender'
]
warn_missing_columns(app, required_app_cols, 'application_train')
app = add_missing_columns(app, required_app_cols, None)

# Limpieza base target y filas mal leídas
app = app.filter(F.col('sk_id_curr').cast('string') != F.lit('SK_ID_CURR'))
app = app.withColumn('target_int', F.col('target').cast('int'))
app = app.filter(F.col('target_int').isin([0, 1]))
app = app.withColumn('target', F.col('target_int')).drop('target_int')

# Features base
app = (
    app
    .withColumn('age_years', F.abs(F.col('days_birth')) / F.lit(365.0))
    .withColumn('flag_days_employed_anomaly', F.when(F.col('days_employed') == 365243, 1).otherwise(0))
    .withColumn('years_employed', F.when(F.col('days_employed') == 365243, None).otherwise(F.abs(F.col('days_employed')) / F.lit(365.0)))
    .withColumn('credit_income_ratio', F.when(F.col('amt_income_total') > 0, F.col('amt_credit') / F.col('amt_income_total')).otherwise(None))
    .withColumn('annuity_income_ratio', F.when(F.col('amt_income_total') > 0, F.col('amt_annuity') / F.col('amt_income_total')).otherwise(None))
    .withColumn('goods_credit_ratio', F.when(F.col('amt_credit') > 0, F.col('amt_goods_price') / F.col('amt_credit')).otherwise(None))
    .withColumn('income_per_family_member', F.when(F.col('cnt_fam_members') > 0, F.col('amt_income_total') / F.col('cnt_fam_members')).otherwise(None))
)

# Categóricas
app = app.withColumn(
    'code_gender',
    F.when(F.upper(F.col('code_gender')) == 'XNA', 'Unknown').otherwise(F.coalesce(F.col('code_gender'), F.lit('Unknown')))
)
string_cols = [f.name for f in app.schema.fields if isinstance(f.dataType, StringType)]
if string_cols:
    app = app.fillna('Unknown', subset=string_cols)

# Numéricas críticas: missing flags + mediana
numeric_fill = [c for c in ['amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price', 'cnt_fam_members'] if c in app.columns]
for c in numeric_fill:
    med = app.approxQuantile(c, [0.5], 0.01)[0]
    app = app.withColumn('flag_missing_{}'.format(c), F.when(F.col(c).isNull(), 1).otherwise(0))
    if med is not None:
        app = app.fillna({c: float(med)})

application_base_clean = app
application_base_clean.createOrReplaceTempView('application_base_clean')
guardar_spark_en_trusted(application_base_clean, 'application_base_clean')
print('application_base_clean listo')

In [ ]:
if bureau is not None:
    b = normalizar_columnas(bureau)
    required_bureau_cols = [
        'sk_id_curr', 'credit_active', 'days_enddate_fact', 'days_credit_enddate',
        'days_credit', 'amt_credit_sum', 'amt_credit_sum_debt', 'amt_credit_sum_overdue', 'credit_day_overdue'
    ]
    warn_missing_columns(b, required_bureau_cols, 'bureau')
    b = add_missing_columns(b, required_bureau_cols, None)

    b = b.withColumn('credit_active_norm', F.lower(F.coalesce(F.col('credit_active'), F.lit('unknown'))))
    b = b.withColumn('is_recent_1y', F.when((F.col('days_credit') >= -365) & (F.col('days_credit') <= 0), 1).otherwise(0))
    b = b.withColumn('missing_days_credit_enddate', F.when(F.col('days_credit_enddate').isNull(), 1).otherwise(0))
    b = b.withColumn('expected_enddate_past', F.when(F.col('days_credit_enddate') < 0, 1).otherwise(0))
    b = b.withColumn('expected_enddate_future', F.when(F.col('days_credit_enddate') >= 0, 1).otherwise(0))
    b = b.withColumn('active_with_enddate_fact', F.when((F.col('credit_active_norm') == 'active') & F.col('days_enddate_fact').isNotNull(), 1).otherwise(0))
    b = b.withColumn('active_expected_enddate_past', F.when((F.col('credit_active_norm') == 'active') & (F.col('days_credit_enddate') < 0), 1).otherwise(0))

    bureau_agg = (
        b.groupBy('sk_id_curr').agg(
            F.count('*').alias('bureau_num_records'),
            F.sum(F.when(F.col('credit_active_norm') == 'active', 1).otherwise(0)).alias('bureau_active_credits'),
            F.sum(F.when(F.col('credit_active_norm') == 'closed', 1).otherwise(0)).alias('bureau_closed_credits'),
            F.sum(F.when(F.col('credit_active_norm').isin('bad debt', 'bad_debt'), 1).otherwise(0)).alias('bureau_bad_debt_credits'),
            F.round(F.sum(F.coalesce(F.col('amt_credit_sum'), F.lit(0.0))), 4).alias('bureau_total_credit_sum'),
            F.round(F.avg(F.col('amt_credit_sum')), 4).alias('bureau_avg_credit_sum'),
            F.round(F.sum(F.coalesce(F.col('amt_credit_sum_debt'), F.lit(0.0))), 4).alias('bureau_total_debt'),
            F.round(F.sum(F.coalesce(F.col('amt_credit_sum_overdue'), F.lit(0.0))), 4).alias('bureau_total_overdue'),
            F.max(F.coalesce(F.col('credit_day_overdue'), F.lit(0))).alias('bureau_max_days_overdue'),
            F.round(F.avg(F.col('days_credit')), 4).alias('bureau_avg_days_credit'),
            F.sum('is_recent_1y').alias('bureau_recent_credits_1y'),
            F.round(F.avg('is_recent_1y'), 6).alias('bureau_pct_recent_credits_1y'),
            F.sum('missing_days_credit_enddate').alias('bureau_missing_days_credit_enddate'),
            F.round(F.avg('missing_days_credit_enddate'), 6).alias('bureau_pct_missing_days_credit_enddate'),
            F.sum('expected_enddate_past').alias('bureau_expected_enddate_past'),
            F.sum('expected_enddate_future').alias('bureau_expected_enddate_future'),
            F.max('active_with_enddate_fact').alias('flag_active_with_enddate_fact'),
            F.round(F.avg('active_with_enddate_fact'), 6).alias('pct_active_with_enddate_fact'),
            F.sum('active_expected_enddate_past').alias('active_expected_enddate_past_count')
        )
    )
else:
    msg = 'WARNING: bureau no disponible; bureau_agg se genera vacía.'
    WARNINGS.append(msg)
    print(msg)
    bureau_agg = application_base_clean.select('sk_id_curr').distinct()

bureau_agg.createOrReplaceTempView('bureau_agg')
guardar_spark_en_trusted(bureau_agg, 'bureau_agg')
print('bureau_agg listo')

In [ ]:
if bureau_balance is not None and bureau is not None:
    bb = normalizar_columnas(bureau_balance)
    bb = add_missing_columns(bb, ['sk_id_bureau', 'months_balance', 'status'], None)

    b_keys = normalizar_columnas(bureau).select('sk_id_bureau', 'sk_id_curr')
    bb = bb.join(b_keys, on='sk_id_bureau', how='inner')
    bb = bb.withColumn('status_norm', F.upper(F.coalesce(F.col('status'), F.lit('X'))))
    bb = bb.withColumn('status_numeric', F.when(F.col('status_norm').isin('0', '1', '2', '3', '4', '5'), F.col('status_norm').cast('int')).otherwise(None))
    bb = bb.withColumn('is_dpd', F.when(F.col('status_norm').isin('1', '2', '3', '4', '5'), 1).otherwise(0))
    bb = bb.withColumn('is_closed', F.when(F.col('status_norm') == 'C', 1).otherwise(0))

    bureau_balance_agg = (
        bb.groupBy('sk_id_curr').agg(
            F.count('*').alias('bb_total_snapshots'),
            F.countDistinct('months_balance').alias('bb_num_months'),
            F.sum(F.col('is_closed')).alias('bb_num_closed'),
            F.sum(F.when(F.col('status_norm') == '0', 1).otherwise(0)).alias('bb_num_status_0'),
            F.sum(F.when(F.col('status_norm').isin('1', '2', '3', '4', '5'), 1).otherwise(0)).alias('bb_num_status_1_5'),
            F.sum(F.when(F.col('status_norm') == 'C', 1).otherwise(0)).alias('bb_num_status_c'),
            F.sum(F.when(F.col('status_norm') == 'X', 1).otherwise(0)).alias('bb_num_status_x'),
            F.round(F.avg(F.col('is_closed')), 6).alias('bb_pct_closed'),
            F.round(F.avg(F.col('is_dpd')), 6).alias('bb_pct_dpd'),
            F.max(F.col('status_numeric')).alias('bb_max_status_numeric'),
            F.max(F.col('months_balance')).alias('bb_recent_month'),
            F.min(F.col('months_balance')).alias('bb_oldest_month')
        )
    )
else:
    msg = 'WARNING: bureau_balance o bureau no disponibles; bureau_balance_agg vacía.'
    WARNINGS.append(msg)
    print(msg)
    bureau_balance_agg = application_base_clean.select('sk_id_curr').distinct()

bureau_balance_agg.createOrReplaceTempView('bureau_balance_agg')
guardar_spark_en_trusted(bureau_balance_agg, 'bureau_balance_agg')
print('bureau_balance_agg listo')

In [ ]:
if previous_application is not None:
    pa = normalizar_columnas(previous_application)
    required_prev = ['sk_id_curr', 'name_contract_status', 'amt_application', 'amt_credit', 'amt_annuity', 'days_decision', 'cnt_payment']
    warn_missing_columns(pa, required_prev, 'previous_application')
    pa = add_missing_columns(pa, required_prev, None)

    pa = pa.withColumn('status_norm', F.upper(F.coalesce(F.col('name_contract_status'), F.lit('UNKNOWN'))))
    pa = pa.withColumn('prev_credit_application_ratio_row', F.when(F.col('amt_application') > 0, F.col('amt_credit') / F.col('amt_application')).otherwise(None))

    previous_application_agg = (
        pa.groupBy('sk_id_curr').agg(
            F.count('*').alias('prev_num_applications'),
            F.sum(F.when(F.col('status_norm') == 'APPROVED', 1).otherwise(0)).alias('prev_approved_count'),
            F.sum(F.when(F.col('status_norm') == 'REFUSED', 1).otherwise(0)).alias('prev_refused_count'),
            F.sum(F.when(F.col('status_norm') == 'CANCELED', 1).otherwise(0)).alias('prev_canceled_count'),
            F.sum(F.when(F.col('status_norm') == 'UNUSED OFFER', 1).otherwise(0)).alias('prev_unused_offer_count'),
            F.round(F.avg(F.col('amt_application')), 4).alias('prev_avg_amt_application'),
            F.round(F.avg(F.col('amt_credit')), 4).alias('prev_avg_amt_credit'),
            F.round(F.avg(F.col('amt_annuity')), 4).alias('prev_avg_amt_annuity'),
            F.round(F.sum(F.coalesce(F.col('amt_credit'), F.lit(0.0))), 4).alias('prev_total_amt_credit'),
            F.round(F.avg(F.col('days_decision')), 4).alias('prev_avg_days_decision'),
            F.min(F.col('days_decision')).alias('prev_min_days_decision'),
            F.round(F.avg(F.col('prev_credit_application_ratio_row')), 6).alias('prev_credit_application_ratio'),
            F.round(F.avg(F.col('cnt_payment')), 4).alias('prev_avg_cnt_payment')
        )
    )

    w_prev = Window.partitionBy('sk_id_curr').orderBy(F.col('days_decision').desc_nulls_last())
    prev_last = (
        pa.select('sk_id_curr', 'status_norm', 'days_decision')
        .withColumn('rn', F.row_number().over(w_prev))
        .filter(F.col('rn') == 1)
        .select('sk_id_curr', F.col('status_norm').alias('prev_last_status'))
    )
    previous_application_agg = previous_application_agg.join(prev_last, on='sk_id_curr', how='left')
else:
    msg = 'WARNING: previous_application no disponible; previous_application_agg vacía.'
    WARNINGS.append(msg)
    print(msg)
    previous_application_agg = application_base_clean.select('sk_id_curr').distinct()

previous_application_agg.createOrReplaceTempView('previous_application_agg')
guardar_spark_en_trusted(previous_application_agg, 'previous_application_agg')
print('previous_application_agg listo')

In [ ]:
if installments_payments is not None:
    inst = normalizar_columnas(installments_payments)
    required_inst = ['sk_id_curr', 'days_entry_payment', 'days_instalment', 'amt_payment', 'amt_instalment']
    warn_missing_columns(inst, required_inst, 'installments_payments')
    inst = add_missing_columns(inst, required_inst, None)

    inst = (
        inst
        .withColumn('payment_delay', F.col('days_entry_payment') - F.col('days_instalment'))
        .withColumn('payment_ratio', F.when(F.col('amt_instalment') > 0, F.col('amt_payment') / F.col('amt_instalment')).otherwise(None))
        .withColumn('flag_late_payment', F.when((F.col('payment_delay') > 0) & F.col('payment_delay').isNotNull(), 1).otherwise(0))
        .withColumn('flag_underpayment', F.when((F.col('payment_ratio') < 1) & F.col('payment_ratio').isNotNull(), 1).otherwise(0))
    )

    installments_agg = (
        inst.groupBy('sk_id_curr').agg(
            F.count('*').alias('inst_num_payments'),
            F.round(F.avg('payment_delay'), 4).alias('inst_avg_payment_delay'),
            F.max('payment_delay').alias('inst_max_payment_delay'),
            F.sum('flag_late_payment').alias('inst_late_payment_count'),
            F.round(F.avg('flag_late_payment'), 6).alias('inst_pct_late_payments'),
            F.round(F.avg('payment_ratio'), 6).alias('inst_avg_payment_ratio'),
            F.round(F.min('payment_ratio'), 6).alias('inst_min_payment_ratio'),
            F.sum('flag_underpayment').alias('inst_underpayment_count'),
            F.round(F.avg('flag_underpayment'), 6).alias('inst_pct_underpayment'),
            F.round(F.sum(F.coalesce(F.col('amt_instalment'), F.lit(0.0))), 4).alias('inst_total_instalment'),
            F.round(F.sum(F.coalesce(F.col('amt_payment'), F.lit(0.0))), 4).alias('inst_total_payment')
        )
    )
else:
    msg = 'WARNING: installments_payments no disponible; installments_agg vacía.'
    WARNINGS.append(msg)
    print(msg)
    installments_agg = application_base_clean.select('sk_id_curr').distinct()

installments_agg.createOrReplaceTempView('installments_agg')
guardar_spark_en_trusted(installments_agg, 'installments_agg')
print('installments_agg listo')

In [ ]:
if pos_cash_balance is not None:
    pos = normalizar_columnas(pos_cash_balance)
    required_pos = ['sk_id_curr', 'months_balance', 'sk_dpd', 'sk_dpd_def', 'name_contract_status', 'cnt_instalment', 'cnt_instalment_future']
    warn_missing_columns(pos, required_pos, 'pos_cash_balance')
    pos = add_missing_columns(pos, required_pos, None)

    pos = pos.withColumn('status_norm', F.upper(F.coalesce(F.col('name_contract_status'), F.lit('UNKNOWN'))))

    pos_cash_agg = (
        pos.groupBy('sk_id_curr').agg(
            F.count('*').alias('pos_num_records'),
            F.round(F.avg('months_balance'), 4).alias('pos_avg_months_balance'),
            F.max('sk_dpd').alias('pos_max_sk_dpd'),
            F.round(F.avg('sk_dpd'), 4).alias('pos_avg_sk_dpd'),
            F.max('sk_dpd_def').alias('pos_max_sk_dpd_def'),
            F.round(F.avg('sk_dpd_def'), 4).alias('pos_avg_sk_dpd_def'),
            F.sum(F.when(F.col('status_norm') == 'COMPLETED', 1).otherwise(0)).alias('pos_completed_count'),
            F.sum(F.when(F.col('status_norm') == 'ACTIVE', 1).otherwise(0)).alias('pos_active_count'),
            F.round(F.avg(F.when(F.col('status_norm') == 'COMPLETED', 1).otherwise(0)), 6).alias('pos_pct_completed'),
            F.round(F.avg('cnt_instalment'), 4).alias('pos_avg_cnt_instalment'),
            F.round(F.avg('cnt_instalment_future'), 4).alias('pos_avg_cnt_instalment_future')
        )
    )
else:
    msg = 'WARNING: pos_cash_balance no disponible; pos_cash_agg vacía.'
    WARNINGS.append(msg)
    print(msg)
    pos_cash_agg = application_base_clean.select('sk_id_curr').distinct()

pos_cash_agg.createOrReplaceTempView('pos_cash_agg')
guardar_spark_en_trusted(pos_cash_agg, 'pos_cash_agg')
print('pos_cash_agg listo')

In [ ]:
if credit_card_balance is not None:
    cc = normalizar_columnas(credit_card_balance)
    required_cc = [
        'sk_id_curr', 'amt_balance', 'amt_credit_limit_actual', 'amt_drawings_current',
        'amt_payment_total_current', 'sk_dpd', 'sk_dpd_def', 'cnt_drawings_current'
    ]
    warn_missing_columns(cc, required_cc, 'credit_card_balance')
    cc = add_missing_columns(cc, required_cc, None)

    cc = cc.withColumn('limit_missing_flag', F.when((F.col('amt_credit_limit_actual').isNull()) | (F.col('amt_credit_limit_actual') <= 0), 1).otherwise(0))
    cc = cc.withColumn('utilization_row', F.when(F.col('amt_credit_limit_actual') > 0, F.col('amt_balance') / F.col('amt_credit_limit_actual')).otherwise(None))

    credit_card_agg = (
        cc.groupBy('sk_id_curr').agg(
            F.count('*').alias('cc_num_records'),
            F.round(F.avg('amt_balance'), 4).alias('cc_avg_balance'),
            F.round(F.max('amt_balance'), 4).alias('cc_max_balance'),
            F.round(F.avg('amt_credit_limit_actual'), 4).alias('cc_avg_credit_limit'),
            F.round(F.max('amt_credit_limit_actual'), 4).alias('cc_max_credit_limit'),
            F.round(F.avg('amt_drawings_current'), 4).alias('cc_avg_drawings'),
            F.round(F.sum(F.coalesce(F.col('amt_drawings_current'), F.lit(0.0))), 4).alias('cc_total_drawings'),
            F.round(F.avg('amt_payment_total_current'), 4).alias('cc_avg_payment_total'),
            F.round(F.sum(F.coalesce(F.col('amt_payment_total_current'), F.lit(0.0))), 4).alias('cc_total_payment'),
            F.max('sk_dpd').alias('cc_max_dpd'),
            F.round(F.avg('sk_dpd'), 4).alias('cc_avg_dpd'),
            F.max('sk_dpd_def').alias('cc_max_dpd_def'),
            F.round(F.avg('sk_dpd_def'), 4).alias('cc_avg_dpd_def'),
            F.round(F.avg('cnt_drawings_current'), 4).alias('cc_avg_cnt_drawings'),
            F.round(F.avg('utilization_row'), 6).alias('cc_utilization_ratio'),
            F.sum('limit_missing_flag').alias('cc_missing_limit_count')
        )
    )
else:
    msg = 'WARNING: credit_card_balance no disponible; credit_card_agg vacía.'
    WARNINGS.append(msg)
    print(msg)
    credit_card_agg = application_base_clean.select('sk_id_curr').distinct()

credit_card_agg.createOrReplaceTempView('credit_card_agg')
guardar_spark_en_trusted(credit_card_agg, 'credit_card_agg')
print('credit_card_agg listo')

In [ ]:
master_dataset = application_base_clean.alias('a')

joins = [
    ('bureau_agg', bureau_agg),
    ('bureau_balance_agg', bureau_balance_agg),
    ('previous_application_agg', previous_application_agg),
    ('installments_agg', installments_agg),
    ('pos_cash_agg', pos_cash_agg),
    ('credit_card_agg', credit_card_agg)
]

for name, dfi in joins:
    cols = [c for c in dfi.columns if c != 'target']
    dfi = dfi.select(*cols).dropDuplicates(['sk_id_curr'])
    master_dataset = master_dataset.join(dfi.alias(name), on='sk_id_curr', how='left')

master_dataset = normalizar_columnas(master_dataset)
master_dataset.createOrReplaceTempView('master_dataset')
guardar_spark_en_trusted(master_dataset, 'master_dataset')

# Validaciones de cardinalidad
app_rows = application_base_clean.count()
master_rows = master_dataset.count()
dup_clients = master_dataset.groupBy('sk_id_curr').count().filter(F.col('count') > 1).count()

print('Rows application_base_clean:', app_rows)
print('Rows master_dataset:', master_rows)
print('Duplicados sk_id_curr en master_dataset:', dup_clients)
print('Columnas finales master_dataset:', len(master_dataset.columns))

null_summary_master_dataset_preview = null_summary(master_dataset, top_n=25)
null_summary_master_dataset_preview.show(25, truncate=False)

In [ ]:
id_cols = [c for c in ['sk_id_curr', 'sk_id_prev', 'sk_id_bureau'] if c in master_dataset.columns]
label_col = 'target'

cat_cols = [
    f.name for f in master_dataset.schema.fields
    if isinstance(f.dataType, StringType) and f.name not in id_cols + [label_col]
]
num_cols = [
    f.name for f in master_dataset.schema.fields
    if isinstance(f.dataType, NumericType) and f.name not in id_cols + [label_col]
]

master_ml_base = master_dataset
if cat_cols:
    master_ml_base = master_ml_base.fillna('Unknown', subset=cat_cols)

stages = []
if cat_cols:
    idx_cols = [c + '_idx' for c in cat_cols]
    ohe_cols = [c + '_ohe' for c in cat_cols]
    indexers = [StringIndexer(inputCol=c, outputCol=idx, handleInvalid='keep') for c, idx in zip(cat_cols, idx_cols)]
    encoder = OneHotEncoder(inputCols=idx_cols, outputCols=ohe_cols, handleInvalid='keep')
    stages.extend(indexers)
    stages.append(encoder)
else:
    idx_cols = []
    ohe_cols = []

if num_cols:
    num_imp_cols = [c + '_imp' for c in num_cols]
    imputer = Imputer(strategy='median', inputCols=num_cols, outputCols=num_imp_cols)
    stages.append(imputer)
else:
    num_imp_cols = []

assembler_inputs = num_imp_cols + ohe_cols
if not assembler_inputs:
    master_ml_base = master_ml_base.withColumn('const_feature', F.lit(1.0))
    assembler_inputs = ['const_feature']

assembler = VectorAssembler(inputCols=assembler_inputs, outputCol='features', handleInvalid='keep')
stages.append(assembler)

pipeline = Pipeline(stages=stages)
model = pipeline.fit(master_ml_base)
master_dataset_ml_ready = model.transform(master_ml_base)
master_dataset_ml_ready = (
    master_dataset_ml_ready
    .withColumn('label', F.col('target').cast('double'))
    .select('sk_id_curr', 'label', 'features')
)

master_dataset_ml_ready.createOrReplaceTempView('master_dataset_ml_ready')
guardar_spark_en_trusted(master_dataset_ml_ready, 'master_dataset_ml_ready')

print('Categorical cols:', len(cat_cols))
print('Numeric cols:', len(num_cols))
print('Encoded cols:', len(ohe_cols))
print('master_dataset_ml_ready listo')

In [ ]:
# Conteos de filas/columnas por dataset
validation_rows = []
for name, dfi in DATASETS.items():
    try:
        validation_rows.append((name, int(dfi.count()), int(len(dfi.columns)), DATASET_PATHS.get(name, '')))
    except Exception:
        validation_rows.append((name, None, None, DATASET_PATHS.get(name, '')))

validation_trusted_outputs = spark.createDataFrame(
    validation_rows,
    schema='dataset string, row_count long, column_count int, output_path string'
)

# Distribución target
if 'target' in master_dataset.columns:
    target_distribution_trusted = (
        master_dataset.groupBy('target')
        .agg(F.count('*').alias('clientes'))
        .orderBy('target')
    )
else:
    target_distribution_trusted = spark.createDataFrame([], 'target int, clientes long')

# Nulos principales
top_nulls = null_summary(master_dataset, top_n=100)

# Validaciones de joins, columnas agregadas y codificación
validation_checks = [
    ('join_validation', 'rows_master_eq_application', float(master_rows == app_rows)),
    ('join_validation', 'duplicated_sk_id_curr_master', float(dup_clients)),
    ('feature_inventory', 'master_columns_count', float(len(master_dataset.columns))),
    ('feature_inventory', 'encoded_categorical_columns', float(len(cat_cols))),
    ('warnings', 'warning_count', float(len(WARNINGS)))
]
validation_checks_df = spark.createDataFrame(validation_checks, 'check_group string, check_name string, check_value double')

# Guardados refined solicitados
guardar_spark_en_refined(validation_trusted_outputs, 'validation_trusted_outputs')
guardar_spark_en_refined(target_distribution_trusted, 'target_distribution_trusted')
guardar_spark_en_refined(top_nulls, 'null_summary_master_dataset')

# Extra útil
if WARNINGS:
    warnings_df = spark.createDataFrame([(w,) for w in WARNINGS], 'warning string')
    guardar_spark_en_refined(warnings_df, 'validation_trusted_warnings')

display(validation_trusted_outputs)
display(target_distribution_trusted)
display(top_nulls.limit(30))

In [ ]:
track_c_rows = [
    ('application_train', 'days_birth', 'edad en dias', 'age_years = abs(days_birth)/365', 'transformar', 'refined/application_base_clean/'),
    ('application_train', 'days_employed=365243', 'anomalia operacional', 'flag_days_employed_anomaly + years_employed limpio', 'flag + limpieza', 'refined/application_base_clean/'),
    ('application_train', 'code_gender=XNA', 'categoria no informativa', "reemplazo por 'Unknown'", 'normalizar categoria', 'refined/application_base_clean/'),
    ('application_train', 'amt_credit/amt_income_total', 'capacidad de endeudamiento', 'credit_income_ratio', 'feature derivada', 'refined/application_base_clean/'),
    ('bureau', 'multiples filas por cliente', 'granularidad por credito', 'agregacion sk_id_curr (bureau_agg)', 'consolidar fuente externa', 'refined/bureau_agg/'),
    ('bureau', 'days_credit_enddate', 'faltantes y vencimientos', 'missing flags + past/future counts', 'calidad + temporalidad', 'refined/bureau_agg/'),
    ('bureau_balance', 'status mensual', 'historial de mora temporal', 'bb_pct_dpd + bb_max_status_numeric + conteos', 'consolidar señal temporal', 'refined/bureau_balance_agg/'),
    ('installments_payments', 'dias y monto pagado vs esperado', 'atraso/subpago', 'payment_delay + payment_ratio + flags', 'behavioral features', 'refined/installments_agg/'),
    ('credit_card_balance', 'saldo y limite', 'utilizacion de credito', 'cc_utilization_ratio', 'feature de uso revolving', 'refined/credit_card_agg/'),
    ('pos_cash_balance', 'dpd mensual', 'mora en POS/Cash', 'pos_max_sk_dpd_def + pos_avg_sk_dpd_def', 'feature de severidad', 'refined/pos_cash_agg/')
]

track_c_decision_matrix = spark.createDataFrame(
    track_c_rows,
    schema='fuente string, variable string, hallazgo_eda string, transformacion_realizada string, decision string, salida_refined string'
)

track_c_decision_matrix.createOrReplaceTempView('track_c_decision_matrix')
guardar_spark_en_refined(track_c_decision_matrix, 'track_c_decision_matrix')

display(track_c_decision_matrix)

## Conclusiones para informe

Este notebook consolidó los tres EDAs funcionales del proyecto (pagos, POS/credit card y bureau/bureau_balance) en un único pipeline de ingeniería de datos para AWS Glue.

Transformaciones implementadas:
- limpieza y validación de `application_train`;
- estandarización de columnas en minúscula;
- manejo de anomalías y faltantes con enfoque robusto para nube;
- agregaciones por cliente en todas las fuentes satélite (`bureau`, `bureau_balance`, `previous_application`, `installments_payments`, `pos_cash_balance`, `credit_card_balance`).

Features creadas:
- razones financieras (capacidad y utilización),
- señales de mora/severidad,
- indicadores de calidad/inconsistencia,
- variables temporales e históricas agregadas por cliente.

Salidas construidas (todas en refined):
- `refined/application_base_clean/`
- `refined/bureau_agg/`
- `refined/bureau_balance_agg/`
- `refined/previous_application_agg/`
- `refined/installments_agg/`
- `refined/pos_cash_agg/`
- `refined/credit_card_agg/`
- `refined/master_dataset/`
- `refined/master_dataset_ml_ready/`
- `refined/validation_trusted_outputs/`
- `refined/target_distribution_trusted/`
- `refined/null_summary_master_dataset/`
- `refined/track_c_decision_matrix/`

Con esto se mantiene trazabilidad técnica completa para Track C sin escribir artefactos en `trusted/`.

Con esto, el flujo queda listo para el siguiente paso: entrenamiento en SparkML, análisis de importancia de variables y comparación de modelos de riesgo de crédito.